<a href="https://colab.research.google.com/github/fmlazohcc/ITAI-1371-ML-Labs/blob/main/Module_13_Lab_Building_ML_Pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 13 Lab - Building ML Pipelines**Objective:** To understand the importance of `scikit-learn` **Pipelines** for creating robust, reproducible, and professional machine learning workflows.**In this lab, you will refactor code from a previous lab into a clean, professional `Pipeline` object.**

## Part 1: Why Use Pipelines?**Concept:** As you've seen, a typical ML workflow involves multiple steps: loading data, cleaning it, splitting it, preprocessing features (scaling, encoding), and finally, training a model. Managing all these steps separately can be messy and error-prone.**Data Leakage:** A major risk of manual preprocessing is **data leakage**. This happens when information from the test set accidentally "leaks" into the training process. For example, if you calculate the mean for scaling using the *entire* dataset before splitting, the model has already "seen" the test data, leading to overly optimistic performance estimates.**A `scikit-learn` Pipeline solves these problems by:**1.  **Encapsulating** all workflow steps into a single object.2.  **Preventing Data Leakage:** It ensures that preprocessing steps are fitted *only* on the training data during cross-validation or when calling `.fit()`.3.  **Improving Reproducibility:** The entire workflow is saved as one object, making it easy to reuse and deploy.

## Part 2: The "Manual" Way (What We Did Before)Let's revisit the Titanic dataset and the steps we took to prepare the data and train a model. This code should look familiar. Notice how many separate objects and steps there are.

In [2]:


import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load the data
df = pd.read_csv(
    "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
)

# Basic feature engineering and cleaning
df["Age"].fillna(df["Age"].median(), inplace=True)
df["Embarked"].fillna(df["Embarked"].mode()[0], inplace=True)
df.drop("Cabin", axis=1, inplace=True)

# Separate features and target
X = df.drop(
    ["Survived", "Name", "Ticket", "PassengerId"],
    axis=1
)

y = df["Survived"]

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Identify feature types
numeric_features = ["Age", "Fare", "SibSp", "Parch"]
categorical_features = ["Pclass", "Sex", "Embarked"]

# Scale numerical features
scaler = StandardScaler()

X_train_scaled_num = scaler.fit_transform(
    X_train[numeric_features]
)

X_test_scaled_num = scaler.transform(
    X_test[numeric_features]
)

# Encode categorical features
encoder = OneHotEncoder(handle_unknown="ignore")

X_train_encoded_cat = encoder.fit_transform(
    X_train[categorical_features]
)

X_test_encoded_cat = encoder.transform(
    X_test[categorical_features]
)

# Combine numerical and categorical features
X_train_processed = np.hstack(
    (
        X_train_scaled_num,
        X_train_encoded_cat.toarray()
    )
)

X_test_processed = np.hstack(
    (
        X_test_scaled_num,
        X_test_encoded_cat.toarray()
    )
)

# Train the model
model = RandomForestClassifier(random_state=42)

model.fit(X_train_processed, y_train)

# Make predictions
y_pred = model.predict(X_test_processed)

print(
    f"Accuracy (Manual Method): "
    f"{accuracy_score(y_test, y_pred):.2%}"
)



# Load datadf = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
# Basic feature engineering and cleaningdf['Age'].fillna(df['Age'].median(), inplace=True)df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)df.drop('Cabin', axis=1, inplace=True)X = df.drop(['Survived', 'Name', 'Ticket', 'PassengerId'], axis=1)y = df['Survived']X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Identify feature typesnumeric_features = ['Age', 'Fare', 'SibSp', 'Parch']categorical_features = ['Pclass', 'Sex', 'Embarked']# Manual Preprocessingscaler = StandardScaler()X_train_scaled_num = scaler.fit_transform(X_train[numeric_features])X_test_scaled_num = scaler.transform(X_test[numeric_features])
# Note: using .transform() here!encoder = OneHotEncoder(handle_unknown='ignore')X_train_encoded_cat = encoder.fit_transform(X_train[categorical_features])X_test_encoded_cat = encoder.transform(X_test[categorical_features])# Combine preprocessed featuresX_train_processed = np.hstack((X_train_scaled_num, X_train_encoded_cat.toarray()))X_test_processed = np.hstack((X_test_scaled_num, X_test_encoded_cat.toarray()))
# Train modelmodel = RandomForestClassifier(random_state=42)model.fit(X_train_processed, y_train)y_pred = model.predict(X_test_processed)

# print(f"Accuracy (Manual Method): {accuracy_score(y_test, y_pred):.2%}")


/tmp/ipykernel_1181/2132082087.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Age"].fillna(df["Age"].median(), inplace=True)
/tmp/ipykernel_1181/2132082087.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', 

Accuracy (Manual Method): 82.68%


## Part 3: The "Pipeline" WayNow, let's do the exact same thing but encapsulate all the preprocessing steps into a single `Pipeline`.**Your Task:** Use `make_pipeline` and `make_column_transformer` to build a complete workflow. This is the modern, professional way to build models in `scikit-learn`.

In [15]:


from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Reload the original dataset
df = pd.read_csv(
    "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
)

# Separate features and target
X = df.drop(
    ["Survived", "Name", "Ticket", "PassengerId", "Cabin"],
    axis=1
)

y = df["Survived"]

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Identify feature types
numeric_features = ["Age", "Fare", "SibSp", "Parch"]
categorical_features = ["Pclass", "Sex", "Embarked"]

# Numerical preprocessing
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

# Categorical preprocessing
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore")
        )
    ]
)

# Combine preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)


# Create the full machine learning pipeline
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(random_state=42))
    ]
)

# Train the pipeline
pipeline.fit(X_train, y_train)

# Make predictions
y_pred = pipeline.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy (Pipeline Method): {accuracy:.2%}")



# Reload the data to start freshdf = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')df.drop(['Cabin', 'Name', 'Ticket', 'PassengerId'], axis=1, inplace=True)X = df.drop('Survived', axis=1)y = df['Survived']X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- ENTER YOUR CODE HERE ---

# 1. Create a pipeline for numeric features#    This pipeline will first impute missing 'Age' values with the median, then scale the features.# numeric_transformer = make_pipeline(#     SimpleImputer(strategy='median'),

#     StandardScaler()# )

# 2. Create a pipeline for categorical features

#    This pipeline will first impute missing 'Embarked' values with the most frequent value, then one-hot encode.# categorical_transformer = make_pipeline(

    # SimpleImputer(strategy='most_frequent'),

    # OneHotEncoder(handle_unknown='ignore')# )

    # 3. Use ColumnTransformer to apply different transformers to different columns# preprocessor = make_column_transformer(#     (numeric_transformer, ['Age', 'Fare', 'SibSp', 'Parch']),

    # (categorical_transformer, ['Pclass', 'Sex', 'Embarked'])# )# 4. Create the final, full pipeline#    This chains the preprocessor and the final model together.

    # final_pipeline = make_pipeline(#     preprocessor,

    # RandomForestClassifier(random_state=42)# )

    # 5. Fit and evaluate the entire pipeline in one step!

    # final_pipeline.fit(X_train, y_train)# y_pred_pipeline = final_pipeline.predict(X_test)

    # print(f"Accuracy (Pipeline Method): {accuracy_score(y_test, y_pred_pipeline):.2%}")


Accuracy (Pipeline Method): 82.68%


## 📝 Reflective Knowledge Check**Instructions:** Answer the following questions in this markdown cell.


1.  **Code Comparison:** Look at the "Manual Way" versus the "Pipeline Way". What are the three biggest advantages you see in using the Pipeline approach?

The three biggest advantages of using the Pipeline approach are organization, prevention of data leakage, and reproducibility. First, the Pipeline combines preprocessing and modeling into one organized object, which makes the code shorter and easier to understand. Second, it helps prevent data leakage because preprocessing steps are fitted only on the training data. Third, it improves reproducibility because the same preprocessing steps are automatically applied whenever the model is used on new data.



2.  **Data Leakage Explained:** In the manual code, we used `scaler.fit_transform()` on the training data but only `scaler.transform()` on the test data. Why was this distinction crucial?

How does the Pipeline automatically handle this for you?


Using scaler.fit_transform() on the training data and only scaler.transform() on the test data was crucial because the scaler must learn the mean and standard deviation only from the training data. If the scaler were fitted on the test data, information from the test set would leak into the training process and make the model’s performance appear better than it really is.

The Pipeline handles this automatically. When final_pipeline.fit() is called, the preprocessing steps are fitted only on the training data. When final_pipeline.predict() is called on the test data, the Pipeline uses the preprocessing values that were already learned from the training data.


3.  **Extending the Pipeline:** Imagine you wanted to add a PCA step to reduce dimensionality *after* scaling and encoding but *before* the RandomForestClassifier. How would you modify your `final_pipeline` object to include this step? (You don't need to write the full code, just describe where you would add `PCA()`.)

To include PCA, I would place PCA() after the preprocessor step and before the RandomForestClassifier in the final_pipeline. This would allow the data to be cleaned, scaled, and encoded first. PCA would then reduce the number of features before the processed data is passed to the classifier.

The order would be:

preprocessor → PCA() → RandomForestClassifier



4.  **Real-World Value:** You are handing your model over to another team to deploy into a web application. Why is giving them the single `final_pipeline` object much safer and more reliable than giving them the 5 separate objects (`scaler`, `encoder`, `model`, etc.) from the manual approach?


Giving another team the single final_pipeline object is safer because it contains the complete workflow in the correct order. The deployment team does not have to remember which missing-value method, scaler, encoder, or model to use separately. This reduces the possibility of applying the wrong preprocessing steps or using them in the wrong order.

The Pipeline also ensures that new web application data is processed in exactly the same way as the training data. This makes deployment more reliable, reproducible, and less likely to produce errors.





**[ENTER YOUR ANSWERS HERE]**